# ACE Supervisor Agent + Endpoint — Production

Registers `SupervisorAgentModel` to Unity Catalog and deploys `ace-chat-endpoint` in a single notebook.
No cross-notebook dependency — `model_version` flows from Cell 3 into Cell 4 within the same session.

`LLM_ENDPOINT` (synthesis) uses Sonnet. `FAST_LLM_ENDPOINT` (classification and session naming) stays on Haiku.

| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Central config — model settings + endpoint settings |
| 3 | Register `SupervisorAgentModel` pyfunc to Unity Catalog |
| 4 | Create or update `ace-chat-endpoint` via REST API |

> **Prod note:** Run cells in order. After Cell 4 prints **READY**, paste the endpoint URL into ACE-Lending `appsettings.json`.


In [ ]:
# ================================================================
# CELL 1 -- Install dependencies
# PURPOSE: Install all Python dependencies needed by this notebook.
#
# Package purposes:
#   mlflow>=2.13.0          : Required for predict_stream() pyfunc streaming.
#     2.13.0+ writes streaming capability flags into the MLmodel artifact
#     at log_model() time. The mlflowserving scoring server reads these
#     flags at line 709 to determine if streaming is supported.
#     Previously we uninstalled pip-mlflow and used Databricks builtin
#     mlflow (for AnyType schema support in infer_signature). mlflow 2.13+
#     includes AnyType support natively, so the builtin is no longer needed.
#   databricks-vectorsearch : VectorSearchClient for SP token pattern.
#   presidio-analyzer       : detects PII entities in page_contexts.
#   presidio-anonymizer     : replaces PII with <PERSON>, <CREDIT_CARD> etc.
#   spacy en_core_web_lg    : Presidio NLP backend (banking-grade NER).
#
# Kernel restarts after %pip install. Re-run from Cell 3.
# ================================================================
%pip install "mlflow>=2.13.0,<3.0.0" databricks-sdk databricks-vectorsearch presidio-analyzer presidio-anonymizer spacy
!python -m spacy download en_core_web_lg
dbutils.library.restartPython()

In [ ]:
# ================================================================
# CELL 2 -- Config
# PURPOSE: Central config -- all endpoint names, LLM names, log table, and routing prompts.
#
#
# All tunable settings live here. Edit this cell and re-run
# Cell 5 to evaluate routing quality without re-registering.
#
# Sub-agent endpoint names:
#   RAG_ENDPOINT_NAME  : ace-rag-agent-endpoint
#     (deployed by 03_rag_agent.ipynb Cell 9)
#   PAGE_ENDPOINT_NAME : ace-lending-context-endpoint
#     (deployed by 04_page_context_agent.ipynb Cell 9)
#   These must match exactly -- the Supervisor calls them by name.
#
# LLM endpoints:
#   LLM_ENDPOINT      : Claude Haiku 4.5 -- synthesis and classification
#     All LLM calls use Haiku. Sub-agents extract all facts; synthesis
#     only combines and formats their output. Haiku is faster, cheaper,
#     and sufficient for this formatting + connecting task.
#
# Monitoring config:
#   TABLE_LOG         : ace_theorem.chat.inference_logs
#     Unified Delta table (created by 01_infrastructure Cell 5).
#   SQL_WAREHOUSE_ID  : b8583158cc1cf9e3
#     The SQL Warehouse the serving container uses to write logs
#     via the SQL Statement Execution REST API. Find under:
#     SQL Warehouses -> your warehouse -> Connection details.
#
# Prompts:
#   CLASSIFICATION_PROMPT : 11-rule routing classifier for Haiku.
#     Covers implicit loan references, approval-type questions,
#     and explicit tiebreakers. Outputs one word (max_tokens=10).
#   SYNTHESIS_PROMPT : 8-rule synthesis prompt.
#     Cites policy facts as [Source N] and loan data as
#     "As per [Tab] tab, ...".
# ================================================================

CATALOG               = "ace_theorem"
SCHEMA                = "chat"

RAG_ENDPOINT_NAME     = "ace-rag-agent-endpoint"
PAGE_ENDPOINT_NAME    = "ace-lending-context-endpoint"
LLM_ENDPOINT          = "databricks-claude-haiku-4-5"
FAST_LLM_ENDPOINT     = "databricks-claude-haiku-4-5"
VS_ENDPOINT_NAME      = "ace-chat-vs-endpoint"
VS_INDEX_NAME         = f"{CATALOG}.{SCHEMA}.policy_documents_index"
MAX_HISTORY_TURNS     = 3
MAX_TOKENS_SYNTHESIS  = 500

# -- Monitoring ------------------------------------------------------
TABLE_LOG             = f"{CATALOG}.{SCHEMA}.inference_logs"
SQL_WAREHOUSE_ID      = "b8583158cc1cf9e3"

# -- MLflow ----------------------------------------------------------
EXPERIMENT_NAME       = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/ace-supervisor-agent"
REGISTERED_MODEL      = f"{CATALOG}.{SCHEMA}.supervisor_agent"
MODEL_ALIAS           = "champion"

CLASSIFICATION_PROMPT = """You are a routing classifier for Vantage Bank's lending AI assistant.

Classify the lending officer's message into exactly one category:

RAG  -- About lending policy, guidelines, procedures, requirements, limits, thresholds, ratios, regulations, exception rules, approval standards, or documentation requirements.
PAGE -- About the specific loan, borrower, collateral, property, rate, terms, values, documents, or application currently being viewed.
BOTH -- Requires both policy knowledge and current loan/application data to answer correctly.
NONE -- Greeting, chitchat, acknowledgment, or unrelated to banking/lending.

Rules:
1. Respond with exactly one word: RAG, PAGE, BOTH, or NONE.
2. Use PAGE if the message asks about any specific current-loan detail, field, number, status, value, person, property, document, or rate.
3. Treat references such as "this loan," "this borrower," "this deal," "this file," "this app," "here," and "current application" as PAGE references.
4. Use RAG if the message asks only about policy, requirements, limits, procedures, regulations, approval standards, or documentation rules and does not depend on current-loan data.
5. Use BOTH if the message asks whether the current loan/application/borrower meets, exceeds, violates, qualifies under, or should be evaluated against a policy, requirement, threshold, ratio, or rule.
6. Use BOTH if the message includes both a current-loan reference and a policy concept, even implicitly.
7. Questions like "can we do this," "does this work," "is this okay," or "can this be approved" should be classified as BOTH when tied to the current loan/application.
8. Use NONE for greetings, thanks, small talk, or non-lending topics.
9. If unsure and the message references only the current loan/application, prefer PAGE.
10. If unsure and the message references both the current loan/application and a rule or requirement, prefer BOTH.
12. Use PAGE if the message asks about the current workflow stage, which queue the loan is in, which role is assigned to act on it, or any of the available actions (Complete, Approve and Submit, Return to Cultivate, Decline, etc.) and what they do.
13. Use PAGE if the message asks "where does this go next?", "who owns this after I approve it?", "what happens if I return to cultivate?", "what is the purpose of this stage?", or any equivalent next-step, routing, or stage-purpose question.
14. Use PAGE for any question about available workflow actions or their downstream routing effects, even when the message does not explicitly name the current loan. Workflow navigation questions are always specific to the loan currently being viewed.
11. Do not explain your reasoning. Output only the label.
"""

SYNTHESIS_PROMPT = """You are a lending assistant for Vantage Bank. Your role is to synthesize sub-agent outputs into one clear, authoritative response for the lending officer.

## Rules
1. Use only the information provided by the sub-agents -- never add outside knowledge, assumptions, or unsupported statements.
2. Treat policy-agent output as the source of truth for policy, requirements, thresholds, and procedures. Treat page-data-agent output as the source of truth for loan, borrower, collateral, and application data.
3. Cite policy facts with [Source N].
4. Cite loan data as: "As per [Tab] tab, ..."
5. When both policy and loan data apply, connect them explicitly:
   "Policy requires X [Source N]. As per [Tab] tab, this loan shows Y. Based on the available information, this [meets / does not meet / may meet / cannot be determined against] that requirement."
6. Only conclude that a loan meets or does not meet a requirement when both the policy requirement and the relevant loan data are explicitly provided by sub-agents. If either is missing or partial, answer only the supported portion and state what remains unverified.
7. If sub-agent outputs conflict, flag the conflict clearly and do not force a definitive conclusion.
8. Be direct, concise, and actionable for a lending officer.

## Response format

**Answer**
- Direct synthesized answer in 1-3 sentences.

**Details**
- **Policy**: supporting requirement, threshold, procedure, or exception [Source N]
- **Loan Data**: exact relevant fact, as per [Tab] tab

**Key Figures** *(omit if none)*
- Policy: exact value [Source N]
- Loan Data: exact value, as per [Tab] tab

**Conflicts / Missing Information** *(omit if none)*
- **Conflict**: describe contradictory sub-agent outputs
- **Missing**: state what additional policy or loan detail is needed to answer definitively

**Sources**
- [Source N]: Document name -- Section name
"""

print("=" * 55)
print("ACE Supervisor Agent -- Config")
print("=" * 55)
print(f"  RAG sub-agent    : {RAG_ENDPOINT_NAME}")
print(f"  Page sub-agent   : {PAGE_ENDPOINT_NAME}")
print(f"  LLM endpoint     : {LLM_ENDPOINT}")
print(f"  Fast LLM         : {FAST_LLM_ENDPOINT}")
print(f"  Log table        : {TABLE_LOG}")
print(f"  SQL warehouse    : {SQL_WAREHOUSE_ID}")
print(f"  Registered model : {REGISTERED_MODEL}")

# -- Endpoint deployment settings ------------------------------------
ENDPOINT_NAME    = "ace-chat-endpoint-v2"
MIN_INSTANCES    = 2
MAX_INSTANCES    = 3
TABLE_LOG        = f"{CATALOG}.{SCHEMA}.inference_logs"
SQL_WAREHOUSE_ID = "b8583158cc1cf9e3"

_raw_url      = spark.conf.get("spark.databricks.workspaceUrl")
WORKSPACE_URL = f"https://{_raw_url}" if not _raw_url.startswith("https://") else _raw_url

print(f"  Endpoint name  : {ENDPOINT_NAME}")
print(f"  SQL warehouse  : {SQL_WAREHOUSE_ID}")
print(f"  Workspace URL  : {WORKSPACE_URL}")
print("=" * 55)

In [ ]:
# ================================================================
# CELL 3 -- Register SupervisorAgentModel as an MLflow pyfunc
# PURPOSE: Register SupervisorAgentModel to Unity Catalog as the production pyfunc.
#
#
# SupervisorAgentModel is a self-contained MLflow PythonModel.
# All constants, prompts, and logic live inside the class so the
# model works in a Databricks Model Serving container without any
# notebook globals.
#
# Class methods:
#
#   load_context(context)
#     Called once when the container starts.
#     Initialises mlflow deployments client.
#     Resolves auth in priority order:
#       1. SP OAuth (MSAL client_credentials flow) -- preferred
#          because the SP has explicit UC grants.
#          Env vars: SP_TENANT_ID, SP_CLIENT_ID, SP_CLIENT_SECRET
#          (injected via ace-secrets scope in 07_serving_endpoint)
#       2. DATABRICKS_TOKEN / DB_TOKEN -- fallback system token
#          (limited UC grants but works for endpoint calls)
#     Reads DATABRICKS_HOST and SQL_WAREHOUSE_ID from env vars
#     for inference logging.
#
#   _call_rag(question)
#     POST to ace-rag-agent-endpoint via mlflow deployments client.
#     Returns {"answer": str, "citations": list}.
#
#   _call_page(question, page_contexts)
#     POST to ace-lending-context-endpoint.
#     Serialises page_contexts list -> JSON string.
#     Returns {"answer": str, "pages_used": list}.
#
#   _classify(question, has_ctx)
#     Claude Haiku intent classification at temp=0.0.
#     Safety overrides: no page ctx -> force RAG; unknown -> BOTH.
#
#   _log_inference(...)
#     Writes one row to inference_logs via SQL Statement Execution
#     REST API in a daemon background thread (non-blocking).
#     User never waits for the log write.
#     Reads SQL_WAREHOUSE_ID from env var (set by 07_serving_endpoint).
#     Silent on failure -- stores last error in self._last_log_error.
#
#   predict(context, model_input)
#     Full orchestration: classify -> route -> synthesize -> log.
#     Accepts DataFrame (MLflow server) or dict (notebook/test).
#     All output list fields (agents_called, citations, pages_used)
#     returned as JSON strings (MLflow schema safe).
#
# resources= block (required for M2M token permissions):
#   - ace-rag-agent-endpoint
#   - ace-lending-context-endpoint
#   - databricks-claude-haiku-4-5
#
# After this cell:
#   - Model registered as ace_theorem.chat.supervisor_agent
#   - Latest version aliased as "champion"
# ================================================================

import json, time, mlflow
from mlflow.models.resources import DatabricksServingEndpoint
from mlflow.pyfunc import ResponsesAgent

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

# -- cloudpickle compatibility shim ----------------------------------
# ResponsesAgent abstract methods have 'request: ResponsesRequest' annotations.
# cloudpickle captures these annotations and stores a module reference.
# In some container MLflow versions ResponsesRequest has been renamed/moved,
# causing AttributeError on load. Injecting a __main__ stub forces cloudpickle
# to embed the class definition so the container resolves it from the pickle.
# Strip ResponsesRequest annotations from ResponsesAgent abstract methods.
# cloudpickle captures the parent class's __annotations__ when serialising
# the subclass.  The serving container's MLflow doesn't export ResponsesRequest
# from mlflow.types.responses, so cloudpickle.load() fails with AttributeError.
# Clearing the annotations before log_model() removes the stale reference.
from mlflow.pyfunc import ResponsesAgent as _RA
for _m in ('predict', 'predict_stream'):
    _fn = _RA.__dict__.get(_m)
    if _fn and hasattr(_fn, '__annotations__'):
        _fn.__annotations__.clear()


class SupervisorAgentModel(mlflow.pyfunc.PythonModel):

    # -- Constants --------------------------------------------
    RAG_ENDPOINT_NAME    = "ace-rag-agent-endpoint"
    PAGE_ENDPOINT_NAME   = "ace-lending-context-endpoint"
    LLM_ENDPOINT         = "databricks-claude-sonnet-4-6"
    FAST_LLM_ENDPOINT    = "databricks-claude-haiku-4-5"
    MAX_TOKENS_SYNTHESIS = 1500
    MAX_HISTORY_TURNS    = 3

    # RAG-only response cache (pure policy questions, no live loan data)
    _RAG_CACHE: dict = {}
    _RAG_CACHE_TTL   = 3600  # 1 hour -- policy docs do not change intra-day

    CLASSIFICATION_PROMPT = """You are a routing classifier for Vantage Bank's lending AI assistant.

Classify the lending officer's message into exactly one category:

RAG  -- About lending policy, guidelines, procedures, requirements, limits, thresholds, ratios, regulations, exception rules, approval standards, or documentation requirements.
PAGE -- About the specific loan, borrower, collateral, property, rate, terms, values, documents, or application currently being viewed.
BOTH -- Requires both policy knowledge and current loan/application data to answer correctly.
NONE -- Greeting, chitchat, acknowledgment, or unrelated to banking/lending.

Rules:
1. Respond with exactly one word: RAG, PAGE, BOTH, or NONE.
2. Use PAGE if the message asks about any specific current-loan detail, field, number, status, value, person, property, document, or rate.
3. Treat references such as "this loan," "this borrower," "this deal," "this file," "this app," "here," and "current application" as PAGE references.
4. Use RAG if the message asks only about policy, requirements, limits, procedures, regulations, approval standards, or documentation rules and does not depend on current-loan data.
5. Use BOTH if the message asks whether the current loan/application/borrower meets, exceeds, violates, qualifies under, or should be evaluated against a policy, requirement, threshold, ratio, or rule.
6. Use BOTH if the message includes both a current-loan reference and a policy concept, even implicitly.
7. Questions like "can we do this," "does this work," "is this okay," or "can this be approved" should be classified as BOTH when tied to the current loan/application.
8. Use NONE for greetings, thanks, small talk, or non-lending topics.
9. If unsure and the message references only the current loan/application, prefer PAGE.
10. If unsure and the message references both the current loan/application and a rule or requirement, prefer BOTH.
12. Use PAGE if the message asks about the current workflow stage, which queue the loan is in, which role is assigned to act on it, or any of the available actions (Complete, Approve and Submit, Return to Cultivate, Decline, etc.) and what they do.
13. Use PAGE if the message asks "where does this go next?", "who owns this after I approve it?", "what happens if I return to cultivate?", "what is the purpose of this stage?", or any equivalent next-step, routing, or stage-purpose question.
14. Use PAGE for any question about available workflow actions or their downstream routing effects, even when the message does not explicitly name the current loan. Workflow navigation questions are always specific to the loan currently being viewed.
11. Do not explain your reasoning. Output only the label.
"""

    SYNTHESIS_PROMPT = """You are a lending assistant for Vantage Bank. Your role is to synthesize sub-agent outputs into one clear, authoritative response for the lending officer.

## Rules
1. Use only the information provided by the sub-agents -- never add outside knowledge, assumptions, or unsupported statements.
2. Treat policy-agent output as the source of truth for policy, requirements, thresholds, and procedures. Treat page-data-agent output as the source of truth for loan, borrower, collateral, and application data.
3. Cite policy facts with [Source N].
4. When presenting loan data, group facts by their source tab and cite the tab ONCE
   as a section sub-header in parentheses -- never repeat the source on every line.
   Format: bold sub-header with tab name in parentheses, then bullet facts beneath.
   Tab names come from the section prefix in the data. Use only the short tab name.
   Tab names by page:
   - Dashboard: Loans In Progress, Worklist, Prospects & Customers
   - CAP: Overview, Loan, Obligor, Rate & Repayment, Collateral, Deposit & Debt,
     Financial Tracking, Exceptions/Approval & Comments, Disbursement & Fees
   - Loan Maintenance: Add Loans, Add Obligors
   Example:
   **Rate Structure** *(Rate & Repayment tab)*
   - Rate 1: Variable at 5.05% indexed to WSJ Prime
   - Rate 2: Fixed at 4.00%
   **Obligor** *(Obligor tab)*
   - AGG Corporation, Texas-based corporation, Borrower
5. When both policy and loan data apply, connect them explicitly:
   "Policy requires X [Source N]. As per Rate & Repayment tab, this loan shows Y. Based on the available information, this [meets / does not meet / may meet / cannot be determined against] that requirement."
6. When both a policy threshold and the corresponding loan value are provided by sub-agents, you MUST apply the threshold to the value and state the result directly -- do not say the data is missing if it was provided. When only partial data is available, state what CAN be determined first (e.g. "The loan amount of  is below/above the  threshold"), then note only the specific fields still missing for a complete assessment. Never refuse to answer entirely when any relevant data is present.
7. If sub-agent outputs conflict, flag the conflict clearly and do not force a definitive conclusion.
8. Be direct, concise, and actionable for a lending officer.
9. Never carry over loan-specific values from conversation history -- loan numbers, loan IDs,
   borrower names, amounts, stages, rates, or statuses must always be read from the current
   page data. History is only used to understand what the officer is asking, never as a
   source of loan facts.
10. Use plain ASCII characters only. Do not use em dashes, en dashes, curly quotes,
   or any Unicode typographic symbols. Use a plain hyphen (-) for dashes, straight
   apostrophes and quotes, and three dots (...) for ellipsis.
11. Never add headings, sections, or labels not listed in the response format below -- do not invent sections like "Missing Information", "Required Data", or "Next Steps".
12. When a compliance question can only be partially answered, present what is determinable as a direct statement, then add a single bullet under Limitations listing only the specific fields still needed -- one sentence maximum.

## Response format
Always produce output in this exact order. Do not add any extra headings.

[Opening paragraph -- NO heading label. 1-3 sentences, direct and definitive.
 Start immediately with the answer. If partially answered, say so at the end.]

**Loan Data** *(omit if no loan data applies)*
Group by source tab. Use the actual tab name as a bold sub-header.
List each fact as a bullet: - Label: Value

**[Tab name]**
- Label: Value
- Label: Value

**Policy Details** *(omit if no policy applies)*
Full explanation with [Source N] cited inline after every fact. Use bullet points
for lists of requirements, exceptions, or conditions.

**Key Figures** *(omit if no specific numbers apply)*
- Metric name: exact value [Source N or tab name]

**Sources**
- [Source N]: Document name -- Section name
- [Tab name] tab: Page name (e.g. Rate & Repayment tab: CAP - 000012)
"""

    # Exclude runtime attrs (self.client, _msal_app, _host, _wh_id) from
    # the cloudpickle.  They contain internal MLflow objects that capture
    # mlflow.types.responses.ResponsesRequest in closures, which fails to
    # deserialise on the serving container.  load_context() re-creates all
    # of these attrs at container start-up before the first request.
    def __getstate__(self):
        return {}

    def __setstate__(self, state):
        pass

    # -- load_context: runs once when the container starts ------------
    def load_context(self, context):
        import logging
        logging.warning("[ACE] load_context called")
        try:
            import mlflow.deployments, os as _os
            from msal import ConfidentialClientApplication
            self.client  = mlflow.deployments.get_deploy_client("databricks")
            self._host   = _os.environ.get("DATABRICKS_HOST", "").rstrip("/")
            self._wh_id  = _os.environ.get("SQL_WAREHOUSE_ID", "")
            _tenant = _os.environ.get("SP_TENANT_ID", "")
            _cid    = _os.environ.get("SP_CLIENT_ID", "")
            _csec   = _os.environ.get("SP_CLIENT_SECRET", "")
            _has_sp = bool(_tenant and _cid and _csec)
            if _has_sp:
                self._msal_app = ConfidentialClientApplication(
                    client_id=_cid,
                    client_credential=_csec,
                    authority=f"https://login.microsoftonline.com/{_tenant}"
                )
            else:
                self._msal_app = None
            logging.warning(f"[ACE] host={bool(self._host)} warehouse={bool(self._wh_id)} sp_vars={_has_sp}")
        except Exception as _lce:
            logging.warning(f"[ACE] load_context FAILED: {type(_lce).__name__}: {_lce}")

    def _get_token(self):
        """Return a valid SP OAuth token via MSAL (cached + auto-refreshed).
        Falls back to DATABRICKS_TOKEN for notebook/dev context.
        """
        import os as _os, logging
        _app = getattr(self, '_msal_app', None)
        if _app is not None:
            result = _app.acquire_token_for_client(
                scopes=["2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default"]
            )
            if "access_token" in result:
                logging.warning("[ACE] token: MSAL OK")
                return result["access_token"]
            logging.warning(f"[ACE] token: MSAL FAILED: {result.get('error','?')}: {str(result.get('error_description',''))[:120]}")
        _tok = _os.environ.get("DATABRICKS_TOKEN", "") or _os.environ.get("DB_TOKEN", "")
        logging.warning(f"[ACE] token: fallback={'set' if _tok else 'EMPTY'}")
        return _tok

    # -- _call_rag: POST to ace-rag-agent-endpoint -------------------
    def _call_rag(self, question: str) -> dict:
        import json as _json
        result    = self.client.predict(
            endpoint = self.RAG_ENDPOINT_NAME,
            inputs   = {"dataframe_records": [{"question": question}]}
        )
        preds     = result.get("predictions", result)
        row       = preds[0] if isinstance(preds, list) and len(preds) > 0 else preds
        raw_cites = row.get("citations", "[]")
        citations = _json.loads(raw_cites) if isinstance(raw_cites, str) else raw_cites
        return {"answer": row.get("answer", ""), "citations": citations}

    # -- _call_page: POST to ace-lending-context-endpoint ----------------
    def _call_page(self, question: str, page_contexts: list) -> dict:
        import json as _json
        result     = self.client.predict(
            endpoint = self.PAGE_ENDPOINT_NAME,
            inputs   = {"dataframe_records": [{
                "question"     : question,
                "page_contexts": _json.dumps(page_contexts)
            }]}
        )
        preds      = result.get("predictions", result)
        row        = preds[0] if isinstance(preds, list) and len(preds) > 0 else preds
        raw_pages  = row.get("pages_used", "[]")
        pages_used = _json.loads(raw_pages) if isinstance(raw_pages, str) else raw_pages
        return {"answer": row.get("answer", ""), "pages_used": pages_used}

    # -- _classify: Claude Haiku intent classification -----------------
    def _classify(self, question: str, has_ctx: bool) -> str:
        user_msg = f"Question: {question}"
        if has_ctx:
            user_msg += "\n(User has page context available)"
        raw = self.client.predict(
            endpoint = self.FAST_LLM_ENDPOINT,
            inputs   = {
                "messages"   : [
                    {"role": "system", "content": self.CLASSIFICATION_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                "max_tokens" : 10,
                "temperature": 0.0
            }
        )["choices"][0]["message"]["content"].strip().upper()
        intent = raw.split()[0] if raw else "BOTH"
        if not has_ctx and intent in ("PAGE", "BOTH"): intent = "RAG"
        if intent not in ("RAG", "PAGE", "BOTH", "NONE"): intent = "BOTH"
        return intent

    # -- _log_inference: background SQL API logging ------------------
    # Single INSERT into inference_logs -- one row per conversation turn.
    # Reads SQL_WAREHOUSE_ID from env var injected by 07_serving_endpoint.
    def _fetch_prior_context(self, host, token, wh_id, upn):
        """Query inference_logs for previous-day turns and summarise with Haiku.
        Called only when history[] is empty (first turn of the day / new session).
        Returns a plain-text summary string, or empty string on any failure.
        """
        import requests as _req, json as _json
        if not host or not token or not wh_id or not upn:
            return ""
        try:
            sql = (
                "SELECT request, response FROM ace_theorem.chat.inference_logs "
                "WHERE requester = :upn "
                "  AND request_date < current_date() "
                "  AND status_response = 'success' "
                "ORDER BY request_time DESC "
                "LIMIT 6"
            )
            r = _req.post(
                f"{host}/api/2.0/sql/statements",
                headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                timeout=12,
                json={"warehouse_id": wh_id, "statement": sql,
                      "parameters": [{"name": "upn", "value": upn, "type": "STRING"}],
                      "wait_timeout": "3s", "format": "JSON_ARRAY"}
            )
            if not r.ok:
                return ""
            data = r.json()
            rows = data.get("result", {}).get("data_array", [])
            if not rows:
                return ""
            pairs = []
            for row in rows:
                if len(row) < 2:
                    continue
                try:
                    req_blob  = _json.loads(row[0]) if isinstance(row[0], str) else row[0]
                    resp_blob = _json.loads(row[1]) if isinstance(row[1], str) else row[1]
                    q = req_blob.get("prompt", "")
                    a = resp_blob.get("answer", "")
                    if q and a:
                        pairs.append("Q: " + q + chr(10) + "A: " + a[:300])
                except Exception:
                    continue
            if not pairs:
                return ""
            raw = (chr(10) + chr(10)).join(reversed(pairs))
            summary_resp = self.client.predict(
                endpoint=self.FAST_LLM_ENDPOINT,
                inputs={
                    "messages": [
                        {"role": "system", "content": (
                            "Summarise the following previous-day conversation snippets "
                            "in 3-5 bullet points. Focus on topics discussed and decisions "
                            "made. Be concise -- this summary will be injected as context "
                            "for a new conversation."
                        )},
                        {"role": "user", "content": raw}
                    ],
                    "max_tokens": 300,
                    "temperature": 0.0
                }
            )
            return summary_resp["choices"][0]["message"]["content"]
        except Exception:
            return ""

    def _log_inference(self, host, token, session_id, intent, answer,
                       latency_ms, prompt, page_contexts, user_identity,
                       history, agents_called, citations, pages_used,
                       wh_id="", error=None):
        import requests as _req, threading, json as _json, uuid as _uuid, os as _os, logging
        from datetime import datetime, timezone as _tz
        def _write():
            logging.warning("[ACE LOG] _write entered")
            try:
                # wh_id param from predict() (pre-resolved from load_context);
                # fall back to env var for notebook context.
                warehouse_id = wh_id or _os.environ.get("SQL_WAREHOUSE_ID", "")
                if not warehouse_id or not host or not token:
                    logging.warning(f"[ACE LOG] SKIP _write: host={bool(host)} token={bool(token)} wh={bool(warehouse_id)}")
                    try: self._last_log_error = f"SKIP: host={bool(host)} token={bool(token)} wh={bool(warehouse_id)}"
                    except: pass
                    return
                now        = datetime.now(_tz.utc)
                request_id = str(_uuid.uuid4())
                status     = "error" if error else "success"
                if isinstance(user_identity, str):
                    try:    uid = _json.loads(user_identity)
                    except: uid = {}
                else:
                    uid = user_identity or {}
                request_blob = _json.dumps({
                    "prompt"       : str(prompt),
                    "history"      : history if not isinstance(history, str) else _json.loads(history),
                    "page_contexts": page_contexts if not isinstance(page_contexts, str) else _json.loads(page_contexts),
                    "user_identity": uid,
                    "session_id"   : str(session_id),
                })
                response_blob = _json.dumps({
                    "answer"       : answer,
                    "intent"       : intent,
                    "agents_called": agents_called,
                    "citations"    : citations,
                    "pages_used"   : pages_used,
                })
                _r = _req.post(
                    f"{host}/api/2.0/sql/statements",
                    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                    timeout=30,
                    json={
                        "wait_timeout" : "0s",
                        "warehouse_id": warehouse_id,
                        "statement": (
                            "INSERT INTO ace_theorem.chat.inference_logs "
                            "(request_date, databricks_request_id, client_request_id, "
                            " request_time, status_code, sampling_fraction, "
                            " execution_duration_ms, request, response, "
                            " served_entity_id, logging_error_codes, requester, "
                            " request_id, type, endpoint, extracted_datetime_str, "
                            " Question, Answer, status_response) "
                            "VALUES (:p1,:p2,:p3,:p4,:p5,:p6,:p7,:p8,:p9,:p10,:p11,:p12,:p13,:p14,:p15,:p16,:p17,:p18,:p19)"
                        ),
                        "parameters": [
                            {"name":"p1",  "value": now.strftime("%Y-%m-%d"),              "type":"DATE"},
                            {"name":"p2",  "value": request_id,                            "type":"STRING"},
                            {"name":"p3",  "value": str(session_id),                       "type":"STRING"},
                            {"name":"p4",  "value": now.strftime("%Y-%m-%d %H:%M:%S"),     "type":"TIMESTAMP"},
                            {"name":"p5",  "value": "500" if error else "200",             "type":"INT"},
                            {"name":"p6",  "value": "1.0",                                 "type":"DOUBLE"},
                            {"name":"p7",  "value": str(latency_ms),                      "type":"LONG"},
                            {"name":"p8",  "value": request_blob[:8000],                   "type":"STRING"},
                            {"name":"p9",  "value": response_blob[:8000],                  "type":"STRING"},
                            {"name":"p10", "value": "ace-chat-endpoint",                   "type":"STRING"},
                            {"name":"p11", "value": str(error)[:2000] if error else "",    "type":"STRING"},
                            {"name":"p12", "value": str(uid.get("upn","unknown")),         "type":"STRING"},
                            {"name":"p13", "value": request_id,                            "type":"STRING"},
                            {"name":"p14", "value": str(intent),                           "type":"STRING"},
                            {"name":"p15", "value": "ace-chat-endpoint",                   "type":"STRING"},
                            {"name":"p16", "value": now.strftime("%Y-%m-%d %H:%M:%S UTC"), "type":"STRING"},
                            {"name":"p17", "value": str(prompt)[:4000],                   "type":"STRING"},
                            {"name":"p18", "value": str(answer)[:4000],                   "type":"STRING"},
                            {"name":"p19", "value": status,                                "type":"STRING"},
                        ]
                    }
                )
                _r.raise_for_status()
                _resp = _r.json()
                _state = _resp.get("status", {}).get("state", "")
                logging.warning(f"[ACE LOG] SQL submit HTTP={_r.status_code} state={_state} wh={warehouse_id[:8]} host={bool(host)}")
                if _state == "FAILED":
                    _err = _resp.get("status", {}).get("error", {}).get("message", "unknown")
                    raise RuntimeError(f"SQL FAILED: {_err}")
            except Exception as _ex:
                logging.warning(f"[ACE LOG] _write exception: {type(_ex).__name__}: {_ex}")
                try: self._last_log_error = f"{type(_ex).__name__}: {_ex}"
                except: pass
        _write()


    # -- _generate_session_name: Haiku session title -----------------
    # Called on the first turn of the day (is_first_turn=True).
    # Returns a 5-7 word title or empty string on any failure.
    def _generate_session_name(self, prompt: str, answer: str) -> str:
        try:
            resp = self.client.predict(
                endpoint=self.FAST_LLM_ENDPOINT,
                inputs={"messages": [{
                    "role": "user",
                    "content": (
                        "In 5-7 words, write a concise title for this conversation. "
                        "Return only the title, no punctuation, no quotes.\n\n"
                        f"User: {prompt[:300]}\nAssistant: {answer[:300]}"
                    )
                }],
                "max_tokens": 30,
                "temperature": 0.0}
            )
            return resp["choices"][0]["message"]["content"].strip()[:80]
        except Exception as _e:
            import logging as _logging
            _logging.getLogger(__name__).error(
                "[session_name] _generate_session_name failed: %s", _e, exc_info=True)
            return ""

    # -- _save_session_name: fire-and-forget INSERT into session_names --
    # Runs in a daemon thread -- never blocks the response stream.
    # Logs success/failure to serving endpoint logs for observability.
    def _save_session_name(self, host, token, wh_id, session_id, requester, name):
        import requests as _req, threading as _threading, logging as _logging
        _log = _logging.getLogger(__name__)
        def _write():
            try:
                if not host or not token or not wh_id:
                    _log.error("[session_name] skipped: missing host/token/wh_id "
                               "(host=%s, wh_id=%s, token_present=%s)", host, wh_id, bool(token))
                    return
                _r = _req.post(
                    f"{host}/api/2.0/sql/statements",
                    headers={"Authorization": f"Bearer {token}",
                             "Content-Type": "application/json"},
                    timeout=5,
                    json={
                        "warehouse_id" : wh_id,
                        "wait_timeout" : "0s",   # fire-and-forget
                        "statement": (
                            "INSERT INTO ace_theorem.chat.session_names "
                            "(session_id, requester, session_name, created_at) VALUES "
                            "(:session_id, :requester, :session_name, current_timestamp())"
                        ),
                        "parameters": [
                            {"name": "session_id",   "value": session_id, "type": "STRING"},
                            {"name": "requester",    "value": requester,  "type": "STRING"},
                            {"name": "session_name", "value": name,       "type": "STRING"},
                        ]
                    }
                )
                if _r.ok:
                    _log.info("[session_name] INSERT accepted (HTTP %s) "
                              "session_id=%s requester=%s name=%s",
                              _r.status_code, session_id, requester, name)
                else:
                    _log.error("[session_name] INSERT rejected (HTTP %s): %s",
                               _r.status_code, _r.text[:500])
            except Exception as _e:
                _log.error("[session_name] _save_session_name exception: %s", _e, exc_info=True)
        _threading.Thread(target=_write, daemon=True).start()


    # -- _cached_rag: cache wrapper for pure RAG intent only -----------
    def _cached_rag(self, question: str) -> dict:
        """Return cached RAG result for pure policy questions (RAG intent only).
        Cache key = MD5 of normalised question. TTL = 1 hour.
        PAGE and BOTH intents always call _call_rag() directly (live data).
        """
        import hashlib as _h, time as _t
        key = _h.md5(question.lower().strip().encode()).hexdigest()
        if key in self._RAG_CACHE:
            result, ts = self._RAG_CACHE[key]
            if _t.time() - ts < self._RAG_CACHE_TTL:
                return result
        result = self._call_rag(question)
        self._RAG_CACHE[key] = (result, _t.time())
        return result

    # -- _flatten_page_contexts: convert tabs -> flat format for page agent --
    def _flatten_page_contexts(self, page_contexts):
        flattened = []
        for page in page_contexts:
            tabs = page.get("tabs", {})
            if not tabs:
                flattened.append(page)
                continue
            all_chunks = []
            all_struct = {}
            for tab_name, tab_data in tabs.items():
                if not isinstance(tab_data, dict):
                    continue
                for chunk in tab_data.get("semantic_chunks", []):
                    all_chunks.append(f"[{tab_name}] {chunk}")
                for k, v in tab_data.get("structured_bucket", {}).items():
                    all_struct[f"{tab_name}_{k}"] = v
            flattened.append({
                "label"            : page.get("label", ""),
                "structured_bucket": all_struct,
                "semantic_chunks"  : all_chunks,
            })
        return flattened

    # -- _call_agents: run RAG + PAGE concurrently for BOTH intent -----
    def _call_agents(self, intent: str, prompt: str, page_contexts: list) -> tuple:
        """Run RAG and PAGE sub-agents concurrently for BOTH intent.
        RAG intent uses the cache; BOTH always calls live (loan data changes daily).
        Returns (rag_result, page_result) -- either may be None.
        """
        from concurrent.futures import ThreadPoolExecutor as _TPE
        if intent == "BOTH":
            with _TPE(max_workers=2) as ex:
                rag_f  = ex.submit(self._call_rag, prompt)
                page_f = ex.submit(self._call_page, prompt, self._flatten_page_contexts(page_contexts))
                return rag_f.result(), page_f.result()
        rag_result  = self._cached_rag(prompt)                                   if intent == "RAG"  else None
        page_result = self._call_page(prompt, self._flatten_page_contexts(page_contexts)) if intent == "PAGE" else None
        return rag_result, page_result


    def predict(self, context=None, model_input=None, params=None):
        return {"output": [{"type": "message", "id": "msg_001", "role": "assistant", "content": [{"type": "output_text", "text": ""}]}]}

    # -- predict_stream: token-level SSE streaming ----------------------
    # Requires route_optimized=True on the served model (07_serving_endpoint
    # Cell 4). Databricks calls this method when the top-level request
    # contains "stream": true (sent by DatabricksSupervisorClient.cs).
    # Classification, RAG, and PAGE block as normal -- only the final
    # Sonnet synthesis call streams tokens back token-by-token.
    def predict_stream(self, request):
        import json as _json, time as _time, os as _os, uuid as _uuid

        def _sanitize_text(t):
            return (
                t
                .replace("—", " - ")
                .replace("–", " - ")
                .replace("‘", "'")
                .replace("’", "'")
                .replace("“", '"' )
                .replace("”", '"' )
                .replace("…", "...")
                .replace(" ", " ")
            )

        if not getattr(self, 'client', None):
            yield {"type": "response.output_text.delta", "item_id": "msg_001", "delta": "", "item": {"type": "response.output_text.delta", "item_id": "msg_001", "output_index": 0, "content_index": 0, "delta": ""}}
            yield {"type": "response.output_item.done", "item_id": "msg_001", "delta": "", "item": {"type": "message", "id": "msg_001", "role": "assistant", "status": "completed", "content": [{"type": "output_text", "text": ""}]}}
            return
        if hasattr(request, "input"):
            _msgs = request.input or []
            ci    = (request.custom_inputs if hasattr(request, "custom_inputs") else {}) or {}
        elif isinstance(request, dict):
            _msgs = request.get("input", [])
            ci    = request.get("custom_inputs", {})
        else:
            _msgs = []
            ci    = {}
        prompt = next(
            ((m.content if hasattr(m, "content") else m.get("content", ""))
             for m in _msgs
             if (m.role if hasattr(m, "role") else m.get("role", "")) == "user"),
            ""
        )
        session_id    = ci.get("session_id", str(_uuid.uuid4()))
        history_raw   = ci.get("history", [])
        page_ctx_raw  = ci.get("page_contexts", [])
        identity_raw  = ci.get("user_identity", {})
        is_first_turn = bool(ci.get("is_first_turn", False))
        history       = _json.loads(history_raw)  if isinstance(history_raw,  str) else history_raw
        page_contexts = _json.loads(page_ctx_raw) if isinstance(page_ctx_raw, str) else page_ctx_raw

        host   = getattr(self, "_host",  None) or _os.environ.get("DATABRICKS_HOST", "").rstrip("/")
        token  = self._get_token()
        wh_id  = getattr(self, "_wh_id", None) or _os.environ.get("SQL_WAREHOUSE_ID", "")
        _start = _time.time()

        # Speculative parallel execution -- mirrors predict() so quality is identical.
        # Classify + RAG + PAGE all start simultaneously; only results matching
        # the resolved intent are passed to synthesis.
        from concurrent.futures import ThreadPoolExecutor as _tpe
        has_ctx     = len(page_contexts) > 0
        _pool       = _tpe(max_workers=3)
        _classify_f = _pool.submit(self._classify, prompt, has_ctx)
        _rag_f      = _pool.submit(self._cached_rag, prompt)
        _page_f     = _pool.submit(self._call_page, prompt, self._flatten_page_contexts(page_contexts)) if has_ctx else None
        _pool.shutdown(wait=False)

        intent = _classify_f.result()

        if intent == "NONE":
            answer = "I can only help with Vantage Bank lending policies and loan information."
            self._log_inference(
                host, token, session_id, "NONE", answer,
                int((_time.time()-_start)*1000),
                prompt=prompt, page_contexts=page_contexts,
                user_identity=_json.loads(identity_raw) if isinstance(identity_raw, str) else identity_raw,
                history=history, agents_called=[], citations=[], pages_used=[], wh_id=wh_id
            )
            _ITEM_ID = "msg_001"
            answer = _sanitize_text(answer)
            yield {"type": "response.output_text.delta", "item_id": _ITEM_ID, "delta": answer, "item": {"type": "response.output_text.delta", "item_id": _ITEM_ID, "output_index": 0, "content_index": 0, "delta": answer}}
            yield {"type": "response.output_item.done", "item_id": _ITEM_ID, "delta": "", "item": {"type": "message", "id": _ITEM_ID, "role": "assistant", "status": "completed", "content": [{"type": "output_text", "text": answer}]}}
            return

        rag_result  = _rag_f.result()                                            if intent in ("RAG", "BOTH") else None
        page_result = _page_f.result() if _page_f is not None and intent in ("PAGE", "BOTH") else None
        agents_called = (["rag_agent"] if rag_result else []) + (["page_context_agent"] if page_result else [])
        citations = rag_result.get("citations", []) if rag_result else []
        if citations:
            _cite_delta = f"__CITE__{_json.dumps(citations)}"
            yield {"type": "response.output_text.delta", "item_id": "msg_001", "delta": _cite_delta, "item": {"type": "response.output_text.delta", "item_id": "msg_001", "output_index": 0, "content_index": 0, "delta": _cite_delta}}

        parts = []
        if not history:
            identity_dict = _json.loads(identity_raw) if isinstance(identity_raw, str) else identity_raw
            upn = identity_dict.get("upn", "")
            if upn:
                prior_summary = self._fetch_prior_context(host, token, wh_id, upn)
                if prior_summary:
                    parts.append(f"Prior-day context summary (for continuity only):\n{prior_summary}")
        if history:
            recent = history[-(self.MAX_HISTORY_TURNS * 2):]
            parts.append("Conversation history:\n" + "\n".join(
                f"{t.get('role','user').capitalize()}: {t.get('content','')}"
                for t in recent
            ))
        if rag_result:  parts.append(f"Policy knowledge:\n{rag_result['answer']}")
        if page_result: parts.append(f"Loan data:\n{page_result['answer']}")

        full_answer_parts = []
        _ITEM_ID = "msg_001"
        _synthesis_inputs = {
            "messages": [
                {"role": "system", "content": self.SYNTHESIS_PROMPT},
                {"role": "user",   "content": (
                    "\n\n".join(parts) +
                    f"\n\nUser question: {prompt}\n\n"
                    "Provide a single clear attributed answer."
                )}
            ],
            "max_tokens" : self.MAX_TOKENS_SYNTHESIS,
            "temperature": 0.0,
        }
        try:
            for chunk in self.client.predict_stream(endpoint=self.LLM_ENDPOINT, inputs=_synthesis_inputs):
                delta = chunk["choices"][0]["delta"].get("content", "")
                if delta:
                    delta = _sanitize_text(delta)
                    full_answer_parts.append(delta)
                    yield {"type": "response.output_text.delta", "item_id": _ITEM_ID, "delta": delta, "item": {"type": "response.output_text.delta", "item_id": _ITEM_ID, "output_index": 0, "content_index": 0, "delta": delta}}
        except Exception as _stream_err:
            # Output guardrail on this endpoint blocks streaming -- fall back to predict().
            if "guardrail" in str(_stream_err).lower() or "streaming mode" in str(_stream_err).lower():
                import logging as _lg
                _lg.warning("[ACE] predict_stream blocked by output guardrail -- falling back to predict()")
                _resp  = self.client.predict(endpoint=self.LLM_ENDPOINT, inputs=_synthesis_inputs)
                _delta = _sanitize_text(_resp["choices"][0]["message"]["content"])
                full_answer_parts.append(_delta)
                yield {"type": "response.output_text.delta", "item_id": _ITEM_ID, "delta": _delta, "item": {"type": "response.output_text.delta", "item_id": _ITEM_ID, "output_index": 0, "content_index": 0, "delta": _delta}}
            else:
                raise
        full_answer = "".join(full_answer_parts)
        citations   = rag_result.get("citations",  []) if rag_result  else []
        pages_used  = page_result.get("pages_used", []) if page_result else []
        identity_dict_log = _json.loads(identity_raw) if isinstance(identity_raw, str) else identity_raw
        self._log_inference(
            host, token, session_id, intent, full_answer,
            int((_time.time()-_start)*1000),
            prompt=prompt, page_contexts=page_contexts,
            user_identity=identity_dict_log, history=history,
            agents_called=agents_called, citations=citations, pages_used=pages_used,
            wh_id=wh_id
        )
        if is_first_turn:
            _name = self._generate_session_name(prompt, full_answer)
            if _name:
                _upn = identity_dict_log.get("upn", "") if isinstance(identity_dict_log, dict) else ""
                self._save_session_name(host, token, wh_id, session_id, _upn, _name)
        yield {"type": "response.output_item.done", "item_id": _ITEM_ID, "delta": "", "item": {"type": "message", "id": _ITEM_ID, "role": "assistant", "status": "completed", "content": [{"type": "output_text", "text": full_answer}]}}



# -- Log and register --------------------------------------------------
# Strategy: save_model locally -> patch MLmodel on disk -> log_artifacts
# -> register_model. This guarantees capabilities.predict_stream: true
# is in the artifact that UC copies. Patching after log_model is too
# late -- register_model may copy from a pre-patch server-side snapshot.

# Register SupervisorAgentModel as a virtual ResponsesAgent subclass.
# Using ABC registration instead of direct inheritance avoids
# ResponsesAgent.__init_subclass__ wrapping predict/predict_stream with
# closures that capture ResponsesRequest -- the root cause of the
# cloudpickle AttributeError on the serving container.
# isinstance(SupervisorAgentModel(), ResponsesAgent) still returns True,
# so MLflow enables streaming identically to direct inheritance.
ResponsesAgent.register(SupervisorAgentModel)

# ---- cloudpickle diagnostic: which class attr contains ResponsesRequest? ----
import cloudpickle as _cp, io as _io
_found = []
for _k, _v in list(SupervisorAgentModel.__dict__.items()):
    _b = _io.BytesIO()
    try:
        _cp.dump(_v, _b)
        if b'ResponsesRequest' in _b.getvalue():
            _found.append(_k)
    except Exception as _ex:
        pass
if _found:
    print(f'ResponsesRequest found in class attrs: {_found}')
else:
    # Also check full instance pickle
    _b2 = _io.BytesIO()
    _cp.dump(SupervisorAgentModel(), _b2)
    if b'ResponsesRequest' in _b2.getvalue():
        print('ResponsesRequest in INSTANCE pickle but not in any class attr')
    else:
        print('OK: ResponsesRequest NOT in pickle -- safe to register')
del _cp, _io, _found
# ---------------------------------------------------------------------------

with mlflow.start_run(run_name="supervisor_agent_registration"):
    model_info = mlflow.pyfunc.log_model(
        artifact_path         = "supervisor_agent",
        python_model          = SupervisorAgentModel(),
        streamable            = True,
        pip_requirements      = ["cloudpickle==3.0.0", "msal"],
        resources = [
            DatabricksServingEndpoint(endpoint_name="ace-rag-agent-endpoint"),
            DatabricksServingEndpoint(endpoint_name="ace-lending-context-endpoint"),
            DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-4-6"),
            DatabricksServingEndpoint(endpoint_name="databricks-claude-haiku-4-5"),
        ],
        registered_model_name = REGISTERED_MODEL,
    )

client_uc = mlflow.tracking.MlflowClient()
versions  = client_uc.search_model_versions(f"name='{REGISTERED_MODEL}'")
if not versions:
    raise RuntimeError("No model versions found after registration.")
version = versions[0].version  # search returns most-recent first

client_uc.set_registered_model_alias(
    name    = REGISTERED_MODEL,
    alias   = MODEL_ALIAS,
    version = version,
)

print(f"Registered : {REGISTERED_MODEL}")
print(f"Version    : {version}")
print(f"Alias      : {MODEL_ALIAS}")
print("Registration complete. Proceed to Cell 8 to verify.")
model_version = version  # available to deploy cell

In [ ]:
# ================================================================
# CELL 4 -- Create or update ace-chat-endpoint
# PURPOSE: Deploy the Supervisor model as the live production endpoint.
#
# Uses the Databricks Model Serving REST API directly (not the SDK)
# for consistency with 03_rag_agent Cell 9 and 04_page_context_agent
# Cell 9 -- all three endpoints use the same served_entities payload
# shape so the pattern is easy to compare and audit.
#
# Environment variables injected into the container:
#   DATABRICKS_HOST    : workspace URL -- needed by _log_inference()
#                        to call the SQL Statement Execution API and
#                        by _call_rag()/_call_page() to build HTTP URLs
#   SQL_WAREHOUSE_ID   : inference log warehouse -- needed by
#                        _log_inference() to target the right warehouse
#   SP_CLIENT_ID       : Service Principal client ID -- used by
#                        load_context() for MSAL OAuth token
#   SP_CLIENT_SECRET   : SP secret -- read from ace-secrets scope
#   SP_TENANT_ID       : Azure AD tenant ID -- read from ace-secrets scope
#
# Why inject SP secrets?
#   The serving container auto-injected DATABRICKS_TOKEN (M2M) has
#   limited Unity Catalog permissions. The SP credentials give the
#   container an OAuth token with full UC grants, including MODIFY on
#   inference_logs for the SQL API write path.
#
# Create vs update strategy:
#   GET /api/2.0/serving-endpoints/<name>
#     404 -> POST to create (first deployment)
#     200 -> PUT /config to update (re-deployment preserves SP ACL)
#
# route_optimized note:
#   route_optimized is an ENDPOINT-LEVEL field -- goes in the POST body
#   alongside "name" and "config", NOT inside served_entities.
#   It is IMMUTABLE after creation -- PUT /config cannot change it.
#   If it needs to change, delete and recreate (Cell 10b).
#   Currently set to False -- the C# package uses MSAL (Azure AD tokens)
#   which are only accepted by the standard workspace URL.
#
# served_entities vs served_models:
#   served_entities is the current API. served_models is deprecated.
#   Field differences: model_name->entity_name, model_version->entity_version
#   workload_type (e.g. "CPU") is required in served_entities.
#
# Polls every 30 s until ready=READY and config_update=NOT_UPDATING.
# Times out after 30 min (60 attempts).
# ENDPOINT_URL is printed below and used in ACE-Lending appsettings.json.
# ================================================================

import time, requests, json

token   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host    = WORKSPACE_URL.rstrip("/")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# -- Served entity definition (shared by create and update) --------
served_entity = {
    "name"                 : "supervisor-agent",
    "entity_name"          : REGISTERED_MODEL,
    "entity_version"       : str(model_version),
    "workload_type"        : "CPU",
    "workload_size"        : "Small",
    "scale_to_zero_enabled": False,
    "environment_vars"     : {
        "DATABRICKS_HOST"  : host,
        "SQL_WAREHOUSE_ID" : SQL_WAREHOUSE_ID,
        "SP_CLIENT_ID"     : "{{secrets/ace-secrets/sp-client-id}}",
        "SP_CLIENT_SECRET" : "{{secrets/ace-secrets/sp-client-secret}}",
        "SP_TENANT_ID"     : "{{secrets/ace-secrets/sp-tenant-id}}",
    }
}

# -- Create or update ----------------------------------------------
check = requests.get(
    f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
    headers=headers
)

if check.status_code == 404:
    print(f"Endpoint '{ENDPOINT_NAME}' does not exist -- creating ...")
    create_payload = {
        "name"           : ENDPOINT_NAME,
        "route_optimized": False,
        "config"         : {
            "served_entities": [served_entity]
        }
    }
    r = requests.post(
        f"{host}/api/2.0/serving-endpoints",
        headers=headers,
        json=create_payload
    )
    r.raise_for_status()
    print("  Create request accepted (route_optimized=False).")
else:
    print(f"Endpoint '{ENDPOINT_NAME}' exists -- updating config ...")
    update_config = {"served_entities": [served_entity]}
    r = requests.put(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
        headers=headers,
        json=update_config
    )
    if not r.ok:
        print(f"  ERROR {r.status_code}: {r.text}")
    r.raise_for_status()
    print("  Update request accepted.")

# -- Poll until READY ----------------------------------------------
print("\nPolling for READY state (max 30 min) ...")
for attempt in range(60):
    state_resp = requests.get(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
        headers=headers
    )
    state_resp.raise_for_status()
    state         = state_resp.json().get("state", {})
    ready_state   = state.get("ready", "")
    config_update = state.get("config_update", "NOT_UPDATING")
    print(f"  [{attempt+1:02d}] ready={ready_state}  config_update={config_update}")
    if config_update == "UPDATE_FAILED":
        raise RuntimeError(f"{ENDPOINT_NAME} UPDATE_FAILED -- check Events tab in Databricks Serving UI for the failure reason.")
    if ready_state == "READY" and config_update in ("NOT_UPDATING", ""):
        break
    time.sleep(30)
else:
    raise RuntimeError(f"{ENDPOINT_NAME} did not become READY within 30 minutes.")

# Standard workspace URL -- works with MSAL (Azure AD) tokens.
ENDPOINT_URL = f"{host}/serving-endpoints/{ENDPOINT_NAME}/invocations"

print("=" * 55)
print(f"Endpoint   : {ENDPOINT_NAME}")
print(f"Model      : {REGISTERED_MODEL} v{model_version}")
print(f"Status     : READY")
print(f"URL        : {ENDPOINT_URL}")
print("=" * 55)
print("ace-chat-endpoint is READY. Paste the URL into ACE-Lending appsettings.json.")
